# Next Best Action (NBA) Recommendation

### Customer Experience Churn Analytics — Banking-AI

This notebook explains how banks use AI to recommend the most relevant next product to a customer:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), familiarity with `scikit-learn` basics, and basic understanding of customer experience and churn analytics terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Explain the customer experience and churn analytics problem and why the chosen classification approach suits this banking workflow.
- Implement an end-to-end classification pipeline on synthetic data and evaluate performance with domain-appropriate metrics.
- Assess whether the model is operationally viable, considering error costs, fairness, and the limitations of synthetic data.

**Estimated time:** 35–45 minutes

**Collection context:** Customer churn prediction, segmentation, lifetime value, and retention strategy in retail banking.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** Instead of spamming all customers with the same offer, banks use "Next Best Action" models. If a customer just started their first job, they might need a Credit Card. If they are older and have high savings, they might need Wealth Management. This improves customer satisfaction and conversion rates.

**Input data used:** Age, Annual Income, Current Balance, Life Stage (Student, Young Professional, Mid-Career, Retired).

**What we predict:** The recommended product (`NBA`).

**ML method used:** Random Forest Classifier (Multi-class).

**Learning goal:** Understand how to build a simple recommendation engine using classification.

**Primary stakeholders:** relationship managers, retention teams, marketing analysts, and branch managers.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Synthetic Customer Profile Dataset

We generate 5,000 customers with different financial backgrounds.

In [ ]:
n = 5000
rng = RNG

age = rng.integers(18, 80, n)
income = rng.normal(60000, 25000, n).clip(15000, 250000)
balance = rng.normal(10000, 15000, n).clip(0, 500000)
tenure_months = rng.integers(1, 120, n)

# Determine Life Stage
life_stages = []
for a in age:
    if a < 25: life_stages.append('Student')
    elif a < 35: life_stages.append('Young Professional')
    elif a < 60: life_stages.append('Mid-Career')
    else: life_stages.append('Retired')

# Define Next Best Action (NBA) Labels
# 0: Savings Account, 1: Credit Card, 2: Mortgage, 3: Wealth Management, 4: No Action
nba = []
for i in range(n):
    if life_stages[i] == 'Student' and balance[i] < 1000:
        nba.append('Savings Account')
    elif life_stages[i] == 'Young Professional' and income[i] > 40000:
        nba.append('Credit Card')
    elif life_stages[i] == 'Mid-Career' and balance[i] > 50000 and income[i] > 80000:
        nba.append('Mortgage')
    elif life_stages[i] == 'Retired' and balance[i] > 100000:
        nba.append('Wealth Management')
    else:
        nba.append('No Action')

df = pd.DataFrame({
    'age': age,
    'income': income,
    'balance': balance,
    'tenure_months': tenure_months,
    'life_stage': life_stages,
    'nba': nba
})

print(df['nba'].value_counts())
df.head()

## Step 4 — Exploratory Data Analysis

EDA

Let's see how Income and Age relate to the recommended action.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='age', y='income', hue='nba', alpha=0.6)
plt.title('Next Best Action by Age and Income')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Scatter plot titled "Next Best Action by Age and Income". The chart highlights spatial separation or clustering patterns in the data.

## Step 5 — Feature Engineering

Prepare features for modelling. Domain-meaningful transformations help the model learn patterns aligned with banking practice.

In [ ]:
# Preprocessing
df_encoded = pd.get_dummies(df, columns=['life_stage'])
X = df_encoded.drop('nba', axis=1)
y = df['nba']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: always predict the majority class ---
baseline_clf = DummyClassifier(strategy='most_frequent', random_state=RANDOM_SEED)
baseline_clf.fit(X_train, y_train)
baseline_pred = baseline_clf.predict(X_test)
baseline_acc = accuracy_score(y_test, baseline_pred)
print(f"Baseline accuracy (majority-class): {baseline_acc:.3f}")
print(f"Any useful model must beat this {baseline_acc:.3f} floor.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"Model Accuracy: {accuracy_score(y_test, y_pred)*100:.1f}%")

In [ ]:
print(classification_report(y_test, y_pred))

importances = pd.Series(model.feature_importances_, index=X.columns)
importances.sort_values().plot(kind='barh', color='#E76F51')
plt.title('Drivers of Next Best Action')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Bar chart titled "Drivers of Next Best Action". The chart ranks features or categories by magnitude to highlight dominant factors.

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

**Summary:**
- The model can accurately predict what product a customer needs based on their **Age**, **Income**, and **Life Stage**.
- **Income** and **Balance** were the strongest drivers for identifying Mortgage and Wealth Management opportunities.

**Banking Context:** These predictions can be integrated into the bank's mobile app or CRM system. When a customer logs in, the "NBA" is displayed as a personalized banner, leading to higher engagement than generic ads.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: score representative cases from the test set ---
print("Operational demonstration — model decision support:\n")
proba = model.predict_proba(X_test)[:, 1]
low_idx  = int(np.argmin(proba))
high_idx = int(np.argmax(proba))
mid_idx  = int(np.argsort(proba)[len(proba) // 2])

for label, idx in [("Low-risk", low_idx), ("Medium-risk", mid_idx), ("High-risk", high_idx)]:
    row = X_test.iloc[idx]
    prob = proba[idx]
    print(f"{label} case  predicted probability: {prob:.1%}")
    print(f"  Features: {dict(row.round(2))}")
    print()

print("A decision-maker would combine these scores with business rules and domain judgement.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end customer experience and churn analytics workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Churn models that rely on demographic features risk discriminatory retention offers.
- Aggressive retention tactics driven by model outputs may erode customer trust.
- Customers should benefit from personalisation, not be manipulated by it.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in customer experience and churn analytics?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.